In [ ]:
import os
from tkinter import filedialog
import pickle
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore", message=r"Passing", category=FutureWarning)
import caiman as cm
from caiman.motion_correction import MotionCorrect

define some parameters

In [ ]:
use_template=False #true if aligning to a template image
template_channel = 'Green' # which channel to perform motion correction; 
multiple=False # true if multiple files per channel
play_movie = True #if want to display motion corrected videos
plot_reg = True # plot average images
save_reg=True # true if want to save motion corrected movie arrays + avg projection tifs.

if multiple:
    green_files = filedialog.askopenfilenames(title='Choose green channel tifs')
    red_files = filedialog.askopenfilenames(title='Choose red channel tifs')

else:
    green_files = [filedialog.askopenfilename(title='Choose green channel tifs')]
    red_files = [filedialog.askopenfilename(title='Choose red channel tifs')]
    path = filedialog.askdirectory(title='Choose save directory')
TSeries = os.path.split(green_files[0])[0]

TSeries

choose which channel to motion correct first as 'template' channel

In [ ]:
if template_channel=='Red':
    mc_files = red_files
    shift_files = green_files
elif template_channel=='Green':
    mc_files = green_files
    shift_files = red_files

In [ ]:
print('motion correcting: '+mc_files[0])
print('applying shifts to: '+shift_files[0])

optional: load a previous mc object params and template image

In [ ]:
load_mc=False #True if you have another mc object with desired params saved. must be saved as a .pkl file with the mc object inside
if load_mc:
    temp_file=filedialog.askopenfilename(title='Choose motion correction pkl file')
    with open(temp_file,'rb') as file:
        mc=pickle.load(file)
    if use_template:
        template=mc.total_template_els
    else:
        template=None
else:
    template=None
template

start cluster

In [ ]:
if 'dview' in locals():
    cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(backend='local', n_processes=None, single_thread=False)

create motion correction object either from previous motion correction object or define parameters

In [ ]:
if load_mc: # if loaded a previous mc object saved
    mc_mcherry = MotionCorrect(mc_files,dview=dview, max_shifts=mc.max_shifts, pw_rigid=mc.pw_rigid,
                      strides=mc.strides, overlaps=mc.overlaps,
                      max_deviation_rigid=mc.max_deviation_rigid,
                      shifts_opencv=mc.shifts_opencv, nonneg_movie=True,
                      border_nan=mc.border_nan)
else: #or create new mc object and define params 
     mc_mcherry = MotionCorrect(mc_files,dview=dview, max_shifts=(40,40), pw_rigid=True,
                      strides=(84,84), overlaps=(12,12),
                      max_deviation_rigid=2, nonneg_movie=True)

mc_mcherry.motion_correct(template=template,save_movie=True)

In [ ]:
mc_mcherry.fname_tot_els

In [ ]:
m_red = cm.load(mc_mcherry.fname_tot_els)
temp_red=m_red.mean(0)
plt.imshow(temp_red)

apply shifts to second color channel + save mmap files

In [ ]:
if mc_mcherry.pw_rigid:
    append = "_els_"
else:
    append="_rig_"

base_name = shift_files[0].split(os.path.sep)[-1].split('.')[-2]+ append

mmap_shifted = mc_mcherry.apply_shifts_movie(shift_files, save_memmap=True, order='F',
                        save_base_name=os.path.join(os.path.split(shift_files[0])[0],base_name))

In [ ]:
mmap_shifted

load movies + calculate average images

In [ ]:
# 
if (mc_mcherry.pw_rigid==True) & (template_channel=='Green'):
    m_green = cm.load(mc_mcherry.fname_tot_els)
    m_red = cm.load(mmap_shifted)
elif (mc_mcherry.pw_rigid==False) & (template_channel=='Green'):
    m_green = cm.load(mc_mcherry.fname_tot_rig)
    m_red = cm.load(mmap_shifted)

elif (mc_mcherry.pw_rigid==True) & (template_channel=='Red'):
    m_red = cm.load(mc_mcherry.fname_tot_els)
    m_green = cm.load(mmap_shifted)
elif (mc_mcherry.pw_rigid==False) & (template_channel=='Red'):
    m_red = cm.load(mc_mcherry.fname_tot_rig)
    m_green=cm.load(mmap_shifted)
    
temp_green=m_green.mean(0)
temp_red=m_red.mean(0)

if play_movie:
    ds_ratio = 0.5
    cm.concatenate([m_green.resize(1, 1, ds_ratio),
                    m_red.resize(1, 1, ds_ratio)], axis=2).play(fr=60, gain=1, magnification=2,
                                                                    offset=0)  # press q to exit

In [ ]:
if plot_reg:
    if template is not None:
        fig, ax = plt.subplots(nrows=1, ncols=3,figsize=(15,5))
        ax[0].imshow(template,cmap='gist_gray')
        ax[0].axis('off')
        ax[0].set_title('920 nm GCaMP Template')
        ax[1].imshow(temp_green,cmap='gist_gray')
        ax[1].set_title('1040 nm GCaMP')
        ax[1].axis('off')
        ax[2].imshow(temp_red,cmap='gist_gray')
        ax[2].set_title('1040 nm mCherry')
        ax[2].axis('off')
        fig.tight_layout()
        
        if save_reg:
            fig.savefig(os.path.join(TSeries, TSeries.split(os.path.sep)[-1] + '_2colorReg.png'))
    else:
        fig, ax = plt.subplots(nrows=1, ncols=2,figsize=(15,5))
        ax[0].imshow(temp_green,cmap='gist_gray')
        ax[0].axis('off')
        ax[0].set_title('Green Channel')
        ax[1].imshow(temp_red,cmap='gist_gray')
        ax[1].set_title('Red Channel')
        ax[1].axis('off')
        fig.tight_layout()
        
        if save_reg:
            fig.savefig(os.path.join(TSeries, TSeries.split(os.path.sep)[-1] + '_2colorReg.png'))

In [ ]:
cm.stop_server(dview=dview) #  stop  server

In [ ]:
if save_reg:
    mchreg_file = os.path.join(TSeries, base_name)

    if hasattr(mc_mcherry,'dview'):
        del(mc_mcherry.dview)
    with open(mchreg_file+'mc.pkl','wb') as file:
        pickle.dump(mc_mcherry,file,protocol=pickle.HIGHEST_PROTOCOL)

    if template is not None: #save original template if exists
        tifffile.imsave(os.path.join(TSeries,os.path.split(temp_file)[-1].split('_')[1])+'_template.tif',template)
    #save mean projection tifs     
    tifffile.imsave(os.path.join(TSeries,red_files[0].split('.')[0].split(os.path.sep)[-1]+append+'.tif'),temp_red)
    tifffile.imsave(os.path.join(TSeries,green_files[0].split('.')[0].split(os.path.sep)[-1]+append+'.tif'),temp_green)

    np.savez(mchreg_file+'movies.npz',m_red=m_red,temp_red=temp_red,m_green=m_green,temp_green=temp_green)

In [ ]:
cm.concatenate([m_green.resize(1, 1, ds_ratio),
                    m_red.resize(1, 1, ds_ratio)], axis=2).play(fr=60, gain=1, magnification=1,
                                                                    offset=0)